In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
# temperature data in celcius
t_c = [0.5, 14.0, 15.0, 28.0, 11.0, 8.0, 3.0, -4.0, 6.0, 13.0, 21.0]

# temperature in unknown units
t_u = [35.7, 55.9, 58.2, 81.9, 56.3, 48.9, 33.9, 21.8, 48.4, 60.4, 68.4]

t_c = torch.tensor(t_c)
t_u = torch.tensor(t_u)

In [ ]:
def model(t_u, w, b):
    return w * t_u + b

def loss_fn(t_p, t_c):
    squared_diffs = (t_p-t_c)**2
    return squared_diffs.mean()

In [ ]:
# represent the parameters as a tensor which is differentiable
params = torch.tensor([1.0,0.0], requires_grad = True)

In [ ]:
params.grad is None

In [ ]:
# generate a function of the parameters
loss = loss_fn(model(t_u, *params), t_c)

# call the backward method to calculate the derivative
# of the function w.r.t. any values that have a grad
loss.backward()

params.grad

In [ ]:
# gradient values are accumulated (added together) with repeated applications
# generate a function of the parameters
loss = loss_fn(model(t_u, *params), t_c)
loss.backward()
params.grad

Notice that the above values are double the original gradient values. We have the loss function evaluated but the gradients have not be reset.

In [ ]:
if params.grad is not None:
    params.grad.zero_() # inplace zero

In [ ]:
def training_loop(n_epochs, learning_rate, params, t_u, t_c):
    for epoch in range(1, n_epochs + 1):
        # ensure gradients are zeroed inplace
        if params.grad is not None:
            params.grad.zero_()
        
        # make the predictions
        t_p = model(t_u, *params)

        # calculate the loss
        loss = loss_fn(t_p, t_c)

        # calculate the gradient of the loss
        loss.backward()

        # update parameters via no_grad context
        # effectively we are updating the computational
        # graph is grad was still allowed because we are
        # essentially defining params as a function of
        # something else. no_grad updates the values, not 
        # the computational graph
        with torch.no_grad():
            params -= learning_rate * params.grad
        
        # print every 500th epock
        if epoch % 500 == 0:
            print('Epoch %d, loss %f' % (epoch, float(loss)))
        
    return params

In [ ]:
t_un = t_u * 0.1

In [ ]:
training_loop(
    n_epochs = 5000,
    learning_rate = 1e-2,
    params = torch.tensor([1.0,0.0], requires_grad = True),
    t_u = t_un,
    t_c = t_c
)

In [ ]:
import torch.optim as optim
dir(optim)

In [ ]:
params = torch.tensor([1.0,0.0], requires_grad = True)
learning_rate = 1e-5
optimize = optim.SGD([params], lr = learning_rate)

In [ ]:
t_p = model(t_un, *params)
loss = loss_fn(t_p, t_c)
loss.backward()

# the update step
optimize.step()

params

In [ ]:
params = torch.tensor([1.0,0.0], requires_grad = True)
learning_rate = 1e-2
optimize = optim.SGD([params], lr = learning_rate)

t_p = model(t_un, *params)
loss = loss_fn(t_p, t_c)

optimize.zero_grad()
loss.backward()

# the update step
optimize.step()

In [ ]:
params

In [ ]:
def training_loop(n_epochs, optimizer, params, t_u, t_c):
    for epoch in range(1, n_epochs+1):
        # get model predictions
        t_p = model(t_u, *params)

        # evaluate loss
        loss = loss_fn(t_p, t_c)

        # ensure zero grad
        optimizer.zero_grad()

        # calculate gradients
        loss.backward()

        # update parameters
        optimizer.step()

        # print results
        if epoch % 500 == 0:
            print('Epoch %d, loss %f' % (epoch, float(loss)))
        
    return params

In [ ]:
params = torch.tensor([1.0,0.0], requires_grad = True)
learning_rate = 1e-2
optimizer = optim.SGD([params], lr = learning_rate)

training_loop(
    n_epochs = 5000, 
    optimizer = optimizer, 
    params = params, 
    t_u = t_un, 
    t_c = t_c
)

In [ ]:
params = torch.tensor([1.0,0.0], requires_grad = True)
learning_rate = 1e-1
optimizer = optim.Adam([params], lr = learning_rate)

training_loop(
    n_epochs = 2000, 
    optimizer = optimizer, 
    params = params, 
    t_u = t_u, # unnormalized input data because Adam adapts well 
    t_c = t_c
)

In [ ]:
n_samples = t_u.shape[0]
# 20% goes to validation
n_val = int(0.2 * n_samples)

# generate 0 through n_samples-1, randomly permutated
shuffled_indices = torch.randperm(n_samples)

# train on all but the last 20%
train_indices = shuffled_indices[:-n_val]

# validate on the final 20%
val_indices = shuffled_indices[-n_val:]

train_indices, val_indices

In [ ]:
# make the train and test data sets

train_t_u = t_u[train_indices]
train_t_c = t_c[train_indices]

val_t_u = t_u[val_indices]
val_t_c = t_c[val_indices]

# normalize
train_t_un  = train_t_u * 0.1
val_t_un = val_t_u * 0.1

In [ ]:
def training_loop(n_epochs, optimizer, params, 
                  train_t_u, val_t_u,
                  train_t_c, val_t_c):
    for epoch in range(1, n_epochs+1):
        # get model predictions on training set
        train_t_p = model(train_t_u, *params)

        # evaluate loss on training set
        train_loss = loss_fn(train_t_p, train_t_c)

        # get model predictions on validation set
        val_t_p = model(val_t_u, *params)

        # evaluate loss on validation set
        val_loss = loss_fn(val_t_p, val_t_c)

        # ensure zero grad
        optimizer.zero_grad()

        # calculate gradients on training data set
        train_loss.backward()

        # update parameters
        optimizer.step()

        # print results
        if epoch % 500 == 0:
            print(f"Epoch {epoch}, Training Loss {train_loss.item():.4f},"
                  f" Validation Loss {val_loss.item():.4f}")
    return params

In [ ]:
params = torch.tensor([1.0,0.0], requires_grad = True)
learning_rate = 1e-2
optimizer = optim.SGD([params], lr = learning_rate)

training_loop(
    n_epochs = 3000, 
    optimizer = optimizer, 
    params = params, 
    train_t_u = train_t_un,
    val_t_u = val_t_un, 
    train_t_c = train_t_c,
    val_t_c = val_t_c
)

In [ ]:
def training_loop(n_epochs, optimizer, params, 
                  train_t_u, val_t_u,
                  train_t_c, val_t_c):
    for epoch in range(1, n_epochs+1):
        # get model predictions on training set
        train_t_p = model(train_t_u, *params)

        # evaluate loss on training set
        train_loss = loss_fn(train_t_p, train_t_c)

        # use context manager to avoid overhead of computational
        # graph when simply evaluating functions and not requiring
        # them in the calculation of gradients
        with torch.no_grad():
            # get model predictions on validation set
            val_t_p = model(val_t_u, *params)

            # evaluate loss on validation set
            val_loss = loss_fn(val_t_p, val_t_c)

            # ensure the gradient is not used
            assert val_loss.requires_grad == False
            
        # ensure zero grad
        optimizer.zero_grad()

        # calculate gradients on training data set
        train_loss.backward()

        # update parameters
        optimizer.step()

        # print results
        if epoch % 500 == 0:
            print(f"Epoch {epoch}, Training Loss {train_loss.item():.4f},"
                  f" Validation Loss {val_loss.item():.4f}")
    return params

In [ ]:
params = torch.tensor([1.0,0.0], requires_grad = True)
learning_rate = 1e-2
optimizer = optim.SGD([params], lr = learning_rate)

training_loop(
    n_epochs = 3000, 
    optimizer = optimizer, 
    params = params, 
    train_t_u = train_t_un,
    val_t_u = val_t_un, 
    train_t_c = train_t_c,
    val_t_c = val_t_c
)